# OVERVIEW

This project aims to create Word2Vec model, training it and evaluating.

Specifically this particular implemented version is based on **Skip-gram** with negative sampling highly inspired by https://web.stanford.edu/~jurafsky/slp3/5.pdf

*Word2Vec* model assigns to each word from vecabulary some n dimentional vector 
$$word \to (a_1,...,a_n)\in \mathbb{R}^n$$
To maintain some semantical and structural properties of words we want similar words to have similar vector representations

We measure similaririty between $w_1$, $w_2$ by dot product


In [ ]:
import numpy as np
import re


# Preprocessing
1. Given raw text firstly we remove newline indicators ("\n") then **split** it into sentences using dots (".") as split points
2. We want to further prepocess text in a way that for every sentence ge get tokens

*... tablespoon of apricot jam and a touch of lemon zest*

We get tokens ['tablespoon', 'of', 'apricot', 'jam', 'and', 'a', 'touch', 'of', 'lemon', 'zest'] *



4. Now we would want to discard interpunction $\{.,";:\}$ 

5. Finally we get list of tokenized sentences 


*Disclaimer: There are better ways for tokinization that include stemming alnd lematization but in this case things are kept simple*

*Aditionally we dont want to delete stopwords(words with low semantic value) because keeping the allows us to better model structure of expressions




In [ ]:
class DataReader:
    
    """Class to read and process text data for Word2Vec training."""

    def __init__(self, file_path):
        self.file_path = file_path

    def read_data(self):
        with open(self.file_path, 'r') as file:
            data = file.read()
            data = data.replace('\n', ' ').lower()
        return data
    
    def process_data(self, data):
        sentences = data.split('.')
        for i in range(len(sentences)):
            sentences[i] = re.sub(r'[^\w\s,;]', '', sentences[i])
            sentences[i] = sentences[i].split()

        return sentences
    
    def get_vocabulary(self, sentences):
        vocab = set()
        for sentence in sentences:
            for word in sentence:
                vocab.add(word)
        return vocab
    
        

# Negative and Positive Samples 

For training a binary classifier we also need negative examples. In fact skip- gram with negative sampling (SGNS) uses more negative examples than positive examples we will use twice as many negative examples as positive ones.


**Positive example** are ones where word(w) and context (c) are likely to occur together We model probality that *c* is context word for target word *w* as
$$P(+|w,c) = \frac{1}{1+\exp(-c \cdot w)}$$


**Negative examples**  are ones where word(w) and context (c) are not likely to occur together we denote probability of them occurng together by
$$P(-|w,c) = \frac{1}{1+\exp(-c \cdot w)}$$



In [ ]:
def sigmoid(x):
        """Stable sigmoid function to prevent overflow issues."""
        x = np.clip(x, -15, 15)
        return 1.0 / (1.0 + np.exp(-x))

# Noise word 
Noise words are choces with propability proportional with their unigram frequency in text 

but we want to set the weith to 3/4 to give rare word sloghtly higher probability

In [ ]:
from collections import Counter

class ExampleFinder:
    def __init__(self, sentences, vocab):
        """
        sentences: list of tokenized sentences, e.g.
                   [["a", "tablespoon", "of", "apricot", "jam"], ...]
        vocab: iterable of words
        """
        self.sentences = sentences
        self.vocab = list(vocab)
        self.word_to_idx = {word: i for i, word in enumerate(self.vocab)}
        self.idx_to_word = {i: word for word, i in self.word_to_idx.items()}
        
        # frequency distribution for negative sampling
        counts = Counter(word for sent in sentences for word in sent if word in self.word_to_idx)
        freqs = np.array([counts[word] for word in self.vocab], dtype=np.float64)
        
        
        freqs = freqs ** 0.75
        self.neg_sampling_probs = freqs / freqs.sum()

    def find_positive_pairs(self, window_size=2):
        """
        Return list of (target_idx, context_idx) positive pairs.
        """
        pairs = []
        for sentence in self.sentences:
            indexed_sentence = [self.word_to_idx[w] for w in sentence if w in self.word_to_idx]
            for i, target_idx in enumerate(indexed_sentence):
                start = max(0, i - window_size)
                end = min(len(indexed_sentence), i + window_size + 1)
                for j in range(start, end):
                    if i != j:
                        context_idx = indexed_sentence[j]
                        pairs.append((target_idx, context_idx))
        return pairs

    def sample_negative_examples(self, target_idx, positive_context_idx, k):
        """
        Sample k negative context indices.
        Avoid target word and the true positive context word.
        """
        negatives = []
        while len(negatives) < k:
            sampled_idx = np.random.choice(len(self.vocab), p=self.neg_sampling_probs)
            if sampled_idx != target_idx and sampled_idx != positive_context_idx:
                negatives.append(sampled_idx)
        return negatives

# Loss function
$$L(w, c_{pos},c_{neg}) = -\log(P(+|w,c_{pos}\cdot P(-|w,c_{neg1})\cdot P(-|w,c_{neg1})))$$

In [ ]:
def logLoss(w, c_pos, c_negs):
    """
    w: shape (embedding_dim,)
    c_pos: shape (embedding_dim,)
    c_negs: shape (k, embedding_dim)
    Return scalar loss value.
    """
    # positive score: c_pos · w
    pos_score = np.dot(c_pos, w)
    pos_prob = sigmoid(pos_score)

    # negative scores: c_neg_i · w
    neg_scores = np.dot(c_negs, w)           # shape (k,)
    neg_probs = sigmoid(-neg_scores)    # because log σ(-c_neg·w)

    # loss = -log σ(c_pos·w) - sum log σ(-c_neg·w)
    eps = 1e-10
    loss = -np.log(pos_prob + eps) - np.sum(np.log(neg_probs + eps))
    return loss

# Embeddings
We initialize embedding for each word as random vector with normal distribution

In [ ]:
def init_embeddings(vocab_size, embedding_dim):
    rng = np.random.default_rng(seed=42)
    # W = target/input embeddings
    W = rng.normal(0, 0.01, size=(vocab_size, embedding_dim))
    # C = context/output embeddings
    C = rng.normal(0, 0.01, size=(vocab_size, embedding_dim))
    return W, C

# Forward pass
We want to 

In [ ]:

class Word2VecSGNS:
    """
    Class to implement Skip-Gram with Negative Sampling (SGNS) model.
    """
    
    def __init__(self, vocab_size, embedding_dim=50, learning_rate=0.025):
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.lr = learning_rate
        self.W, self.C = init_embeddings(vocab_size, embedding_dim)
    

    def forward(self, target_idx, pos_context_idx, neg_context_indices):
        """
        Compute forward pass and loss for one training example.
        
        Returns:
            loss
            cached values needed for update
        """
        w = self.W[target_idx]                    # shape (d,)
        c_pos = self.C[pos_context_idx]          # shape (d,)
        c_negs = self.C[neg_context_indices]     # shape (k, d)
        pos_score = np.dot(c_pos, w)             # scalar
        neg_scores = np.dot(c_negs, w)          # shape (k,)
        loss = logLoss(w, c_pos, c_negs)

        cache = {
            "w": w.copy(),
            "c_pos": c_pos.copy(),
            "c_negs": c_negs.copy(),
            "pos_score": pos_score,
            "neg_scores": neg_scores,
            "target_idx": target_idx,
            "pos_context_idx": pos_context_idx,
            "neg_context_indices": neg_context_indices
        }
        return loss, cache

    def update(self, cache):
        """
        Apply SGD update
        """
        target_idx = cache["target_idx"]
        pos_context_idx = cache["pos_context_idx"]
        neg_context_indices = cache["neg_context_indices"]

        w = cache["w"]
        c_pos = cache["c_pos"]
        c_negs = cache["c_negs"]

        pos_score = cache["pos_score"]
        neg_scores = cache["neg_scores"]

    
        # dL/dc_pos = [σ(c_pos·w) - 1] w
        grad_c_pos = (self.sigmoid(pos_score) - 1.0) * w

        # dL/dc_neg_i = [σ(c_neg_i·w)] w
        grad_c_negs = self.sigmoid(neg_scores)[:, None] * w[None, :]

        # dL/dw = [σ(c_pos·w)-1] c_pos + sum_i σ(c_neg_i·w) c_neg_i
        grad_w = (self.sigmoid(pos_score) - 1.0) * c_pos \
                 + np.sum(self.sigmoid(neg_scores)[:, None] * c_negs, axis=0)

        # SGD updates
        self.C[pos_context_idx] -= self.lr * grad_c_pos
        self.C[neg_context_indices] -= self.lr * grad_c_negs
        self.W[target_idx] -= self.lr * grad_w

    def train_one_example(self, target_idx, pos_context_idx, neg_context_indices):
        loss, cache = self.forward(target_idx, pos_context_idx, neg_context_indices)
        self.update(cache)
        return loss

    def get_word_vector(self, word_idx, combine="sum"):
        """
        Common choices:
        - 'W'   -> use only target embeddings
        - 'C'   -> use only context embeddings
        - 'sum' -> W + C
        """
        if combine == "W":
            return self.W[word_idx]
        elif combine == "C":
            return self.C[word_idx]
        elif combine == "sum":
            return self.W[word_idx] + self.C[word_idx]
        else:
            raise ValueError("combine must be one of {'W', 'C', 'sum'}")

In [11]:
def train_word2vec(sentences, vocab, embedding_dim=50, window_size=2, k=5, epochs=5, learning_rate=0.025):
    finder = ExampleFinder(sentences, vocab)
    pairs = finder.find_positive_pairs(window_size=window_size)

    model = Word2VecSGNS(
        vocab_size=len(finder.vocab),
        embedding_dim=embedding_dim,
        learning_rate=learning_rate
    )

    for epoch in range(epochs):
        np.random.shuffle(pairs)
        total_loss = 0.0

        for target_idx, pos_context_idx in pairs:
            neg_context_indices = finder.sample_negative_examples(
                target_idx, pos_context_idx, k
            )
            loss = model.train_one_example(target_idx, pos_context_idx, neg_context_indices)
            total_loss += loss

        avg_loss = total_loss / len(pairs)
        print(f"Epoch {epoch+1}/{epochs}, avg loss = {avg_loss:.4f}")

    return model, finder

# Training Data

Data comes from Australian Broadcasting Commission 2006
http://www.abc.net.au/

Rural News    http://www.abc.net.au/rural/news/

In [14]:
sentences = [
    ["lemon", "a", "tablespoon", "of", "apricot", "jam", "a", "pinch"],
    ["apricot", "jam", "is", "sweet"],
    ["lemon", "jam", "is", "sour"],
    ["a", "tablespoon", "of", "jam"]
]

vocab = sorted(set(word for sent in sentences for word in sent))

model, finder = train_word2vec(
    sentences,
    vocab,
    embedding_dim=20,
    window_size=2,
    k=4,
    epochs=50,
    learning_rate=0.05
)

Epoch 1/50, avg loss = 3.4657
Epoch 2/50, avg loss = 3.4651
Epoch 3/50, avg loss = 3.4643
Epoch 4/50, avg loss = 3.4631
Epoch 5/50, avg loss = 3.4595
Epoch 6/50, avg loss = 3.4523
Epoch 7/50, avg loss = 3.4357
Epoch 8/50, avg loss = 3.4026
Epoch 9/50, avg loss = 3.3302
Epoch 10/50, avg loss = 3.2102
Epoch 11/50, avg loss = 3.0416
Epoch 12/50, avg loss = 2.8378
Epoch 13/50, avg loss = 2.7166
Epoch 14/50, avg loss = 2.6184
Epoch 15/50, avg loss = 2.5494
Epoch 16/50, avg loss = 2.5601
Epoch 17/50, avg loss = 2.4919
Epoch 18/50, avg loss = 2.4940
Epoch 19/50, avg loss = 2.4544
Epoch 20/50, avg loss = 2.4814
Epoch 21/50, avg loss = 2.4808
Epoch 22/50, avg loss = 2.4255
Epoch 23/50, avg loss = 2.3872
Epoch 24/50, avg loss = 2.3833
Epoch 25/50, avg loss = 2.4452
Epoch 26/50, avg loss = 2.3697
Epoch 27/50, avg loss = 2.3921
Epoch 28/50, avg loss = 2.3404
Epoch 29/50, avg loss = 2.3783
Epoch 30/50, avg loss = 2.2908
Epoch 31/50, avg loss = 2.3401
Epoch 32/50, avg loss = 2.3528
Epoch 33/50, avg 

In [27]:
dr = DataReader("rural.txt")
sentences= dr.process_data(dr.read_data())[:1000]
vocab = dr.get_vocabulary(sentences)

model, finder = train_word2vec(
    sentences,
    vocab,
    embedding_dim=20,
    window_size=2,
    k=2,
    epochs=50,
    learning_rate=0.05
)

Epoch 1/50, avg loss = 2.0742
Epoch 2/50, avg loss = 2.0286
Epoch 3/50, avg loss = 1.9441
Epoch 4/50, avg loss = 1.8577
Epoch 5/50, avg loss = 1.7783
Epoch 6/50, avg loss = 1.7071
Epoch 7/50, avg loss = 1.6424
Epoch 8/50, avg loss = 1.5855
Epoch 9/50, avg loss = 1.5315
Epoch 10/50, avg loss = 1.4869
Epoch 11/50, avg loss = 1.4443
Epoch 12/50, avg loss = 1.4083
Epoch 13/50, avg loss = 1.3690
Epoch 14/50, avg loss = 1.3410
Epoch 15/50, avg loss = 1.3109
Epoch 16/50, avg loss = 1.2844
Epoch 17/50, avg loss = 1.2598
Epoch 18/50, avg loss = 1.2389
Epoch 19/50, avg loss = 1.2181
Epoch 20/50, avg loss = 1.1963
Epoch 21/50, avg loss = 1.1837


KeyboardInterrupt: 

# Discussion, possible improvements 
1. Skip-gram vs CBOW(Continous bag of Words) 
Both of these approaches tfind connection with occurances of given word and context words toghether. However they do this in completely different ways

In both models we want to create vectors for words from our vocabulary but in c CBOW we want to predict target word from context words

2. Different size of context might catch some more complex dependencies in language but will mke training longer 
